# [실습] LLM API, Structured Output & Tool Calling

LangChain을 활용하여 다양한 LLM API를 호출하고, Tool Calling과 Structured Output을 실습합니다.

## 학습 목표

- `init_chat_model`로 LLM(GPT-5.2) 초기화 방법 학습
- Structured Output (Pydantic 바인딩)으로 구조화된 데이터 생성
- `@tool` 데코레이터로 커스텀 도구를 정의하고 사용하기
- Tool Calling 메커니즘: `tool_calls` 메시지 구조 이해


## 환경 설정

In [1]:
%pip install langchain langchain-openai langchain-core -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

# load_dotenv('.env', override=True)
load_dotenv()

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 설정 완료!')
else:
    print('OpenAI API 키가 설정되지 않았습니다. .env 파일을 확인하세요.')

OpenAI API 키 설정 완료!


---
## 1. LLM 초기화하기

LangChain 1.x에서는 `init_chat_model`을 사용하여 다양한 LLM을 통일된 인터페이스로 초기화할 수 있습니다.

### 1-1. init_chat_model (권장 방식)

`init_chat_model`은 모델 이름으로부터 자동으로 Provider를 감지합니다.

In [3]:
from langchain.chat_models import init_chat_model

# GPT-5.2 초기화 (OpenAI)
gpt = init_chat_model('gpt-5.2', max_tokens=16384,
    # timeout = 60, # 60초 이상 응답이 없으면 종료
    # max_retries = 6, # 실패 시 재시도 횟수 (기본값 6, 비활성화: 1)
    )
print(f'LLM 모델 초기화 완료!')
print(f'모델 이름: {gpt.model_name}')

LLM 모델 초기화 완료!
모델 이름: gpt-5.2


### 1-2. Provider별 클래스 지정

특정 Provider의 모델을 사용하는 경우, 클래스를 직접 지정하기도 합니다.   
- `ChatOpenAI, ChatGoogleGenerativeAI, ChatOllama, ChatHuggingFace` 등

In [4]:
from langchain_openai import ChatOpenAI

gpt_direct = ChatOpenAI(model='gpt-5.2', max_tokens=16384)
gpt_direct

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-5.2', 'release_date': '2025-12-11', 'last_updated': '2025-12-11', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002967E8F3950>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002967E8F8090>, root_client=<openai.OpenAI object at 

### 1-3. LLM에 메시지 전달하기   

LangChain의 모든 모듈은 `invoke()`를 통해 실행합니다.   
LLM을 invoke하면 `AIMessage` 클래스를 return합니다.

In [5]:
# 1. 문자열을 통한 입력
response = gpt.invoke('안녕하세요! 한 문장으로 자기 소개 부탁드립니다.')
print(f'GPT 응답: {response.text}')
response


GPT 응답: 저는 사용자의 질문에 답하고 글쓰기·번역·아이디어 정리·문제 해결을 돕는 AI 비서입니다.


AIMessage(content='저는 사용자의 질문에 답하고 글쓰기·번역·아이디어 정리·문제 해결을 돕는 AI 비서입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 18, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-Du74aVxBzOmyxRtgut0G0Rc8Ml7Wt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef754-6630-7553-94b9-9d75d56a0ebe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 34, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

`stream`은 토큰이나 이벤트 단위로 결과물을 출력합니다.

In [6]:
for chunk in gpt.stream('AI Agent의 핵심 구성 요소를 세 단어로 표현해보세요.'):
    print(chunk.text, end='', flush=True)

- **지각(Perception)**  
- **추론(Reasoning)**  
- **행동(Action)**

위에서 입력한 문자열은 내부적으로 `HumanMessage` 클래스로 LLM에 전달됩니다.    

보다 자세한 입력을 위해, 랭체인의 내부 클래스를 직접 활용합니다.

### 1-4. Message List를 통한 입력

In [7]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# System Message: 첫 번째로 들어가는 지시사항이나 지침, Human Message보다 우선순위가 높음

msg_list = [
    SystemMessage('답변을 마친 후에는, 질문 형식으로 다음 작업에 대해 제안하세요.'),
    HumanMessage('Agent와 LLM의 차이에 대해 한 문장으로 설명해주세요.')
]

stream = gpt.stream(msg_list)

for s in stream:
    print(s.text, end='', flush=True)

LLM은 입력을 받아 텍스트를 생성하는 모델 자체인 반면, Agent는 LLM을 포함해 도구·메모리·계획·실행 루프를 결합하여 목표를 스스로 수행하는 시스템입니다.

Agent와 LLM의 차이를 예시(예: 고객지원, 코딩, 데이터분석)로도 한두 가지 들어 설명해드릴까요?

---
## 2. Prompt Template과 Chain   

프롬프트 템플릿을 사용하면 입력 변수를 포함한 재사용 가능한 프롬프트를 작성할 수 있습니다.

`ChatPromptTemplate`은 시스템 프롬프트와 사용자 프롬프트를 구분하여 설정합니다.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
    ('system', '''2025-2026년까지의 최신 지식을 최대한 활용하여 답변하세요. 
간결하게 3문장 이내로 답변하세요. 이모지를 많이 사용하세요. 최소 10개'''),
    ('human', '{question}') # Field가 {question} 
])

# 프롬프트와 LLM을 | (파이프)로 연결하면 체인(Chain)이 됩니다
chain = prompt | gpt # LCEL방식 

response = chain.invoke('LLM의 최근 컨텍스트 윈도우는 어디까지 확장되고 있나요?')

print(response.text)

2025~2026 기준으로 LLM 컨텍스트 윈도우는 상용·최신 모델에서 **수십만 토큰(예: 128k~200k+)**까지가 흔해졌고, 일부는 **100만 토큰(1M)급**까지 확장되는 흐름입니다. 📈🧠📚🧩📝🔍⚙️🚀💾🧵🌐  
다만 **1M급은 비용·지연·정확도(장기 주의/검색 혼합 필요)** 이슈로, 실제 제품에선 **긴 컨텍스트 + RAG/메모리** 조합이 주류입니다. 🛠️⏱️💰🎯📉🔗📌🗂️🧠✨


입력 변수가 두 개 이상일 때는 dict 형태로 전달합니다.

In [10]:
analysis_prompt = ChatPromptTemplate([
    ('system', '''당신은 LLM Agent 설계 전문가입니다.
주어진 문제를 해결하는 Agent의 Tool 목록과 주요 Failure Scenario를 각각 {num}개 이내의 Bullet으로 설명하세요.'''),
    ('human', '문제: {problem}') # Field가 {num}, {problem}
])

chain = analysis_prompt | gpt

# 단일 호출
response = chain.invoke({'problem': '사내 이메일 자동화 에이전트', 'num': '3'})
print(response.text)

## Tool 목록 (3개 이내)

- **Email API Tool (Microsoft Graph / Gmail API)**
  - 메일 조회(조건 검색), 스레드/대화 ID 추적, 라벨/폴더 이동, 읽음 처리, 답장/전달/초안 생성, 첨부 다운로드/업로드 등 이메일 실행 작업 담당

- **Policy & Compliance Tool (DLP/PII 탐지 + 권한/승인 워크플로)**
  - 사내 규정(대외발신 제한, 보존정책, 수신자 도메인 제한), 민감정보(PII/PHI/재무정보) 탐지/마스킹, 고위험 발신 시 상신(결재) 또는 Human-in-the-loop 승인 게이트

- **Enterprise Knowledge & Action Tool (RAG + 업무시스템 연동)**
  - 사내 위키/규정/FAQ/템플릿을 검색(RAG)해 답변 초안 구성, 필요 시 Jira/ServiceNow/CRM/ERP 등 티켓 생성·상태 조회·코멘트 추가 같은 후속 액션 수행


## 주요 Failure Scenario (3개 이내)

- **잘못된 수신자/스레드에 발신 (Mis-send / Thread Hijack)**
  - 유사한 이름·메일주소 자동완성, 스레드 컨텍스트 오인, “전체답장” 오동작으로 내부/외부에 민감정보가 잘못 전송

- **규정 위반 또는 정보 유출 (Compliance/DLP bypass)**
  - 첨부파일/인용문 내 민감정보를 탐지 못하거나, 외부 도메인 발신 제한·보존정책을 무시한 자동 발송으로 보안 사고 발생

- **의도 오해로 인한 업무 자동처리 오류 (Incorrect automation)**
  - 사용자의 요청(“승인 메일 보내줘”, “연기 요청”)을 잘못 해석해 반대 내용 발신, 티켓/주문/계정 권한 변경 등 되돌리기 어려운 액션을 실행하여 운영 장애 유발



또한 `batch()`를 사용하면 여러 입력을 한 번에 처리할 수 있습니다.

In [12]:
# batch 호출: 여러 입력을 한 번에 처리
problems = [
    {'problem': '논문 요약 에이전트', 'num': '3'},
    {'problem': '코드 리뷰 에이전트', 'num': '3'},
]
results = chain.batch(problems) # 병렬 처리 시 batch로 처리

for i, result in enumerate(results):
    print(f'--- {problems[i]["problem"]} ---')
    print(result.text)
    print()

--- 논문 요약 에이전트 ---
### Tool 목록 (3개 이내)
- **PDF/문서 파서 + 레이아웃 보존 추출기**: PDF에서 본문/표/그림 캡션/참고문헌을 분리 추출하고 섹션(초록, 서론, 방법, 결과, 결론)을 구조화.
- **인용/메타데이터 조회 도구 (Crossref/Semantic Scholar 등)**: 논문 제목/저자/연도/저널, DOI, 초록, 인용수, 관련 후속연구를 가져와 요약의 신뢰도와 맥락 보강.
- **검증/근거 하이라이트 도구**: 요약 문장별로 원문 근거(문단/페이지/문장)를 링크하거나 하이라이트로 매핑해 “요약-근거” 추적 가능하게 함.

### 주요 Failure Scenario (3개 이내)
- **환각 요약(근거 없는 주장 생성)**: 원문에 없는 실험 설정/수치/결론을 만들어내거나 과장해서 서술(특히 결과 수치, SOTA 주장).
- **구조/레이아웃 파싱 실패로 인한 내용 누락**: 2단 PDF, 수식·표·그림 캡션이 깨져 핵심 실험 조건/결과가 요약에서 빠지거나 잘못 연결됨.
- **요약 목표 불일치**: 사용자가 원하는 수준(초간단/한 페이지/리뷰용/방법 재현용)이나 관점(기여점 중심 vs 한계 중심)과 다른 형식·깊이로 요약해 유용성이 떨어짐.

--- 코드 리뷰 에이전트 ---
### Tool 목록 (3개 이내)
- **VCS/PR 인터페이스 도구 (GitHub/GitLab API)**
  - PR 메타데이터(작성자, 라벨, 변경 파일, 커밋), 라인 코멘트/리뷰 제출, 리뷰 상태 변경, 체크(run) 코멘트 게시.
- **Diff/AST/정적분석 도구**
  - Diff 파서(파일/함수/라인 단위), 언어별 AST 파싱, 린터/타입체커/보안 스캐너(SAST) 결과 수집 및 요약.
- **테스트/빌드 실행 도구 (CI 연동 또는 샌드박스 실행)**
  - 단위/통합 테스트 실행, 커버리지 확인, 재현 가능한 최소 실행 환경에서 빌드 검증 및 실패 로그 수집.

### 주요 Failure Scenario (3

---
## 3. Structured Output (구조화된 출력)

LLM의 결과물을 모듈화하고 활용하기 위해서는 보통 출력 텍스트의 후처리가 필요합니다.   

LLM의 출력을 구조화하는 방법에 대해 알아보겠습니다.   

Pydantic은 파이썬을 위한 데이터 클래스로, 

`with_structured_output()`을 사용하면 LLM이 지정된 스키마에 맞는 JSON을 출력하고, 자동으로 Pydantic 객체로 변환됩니다.

### 3-1. 기본 Structured Output

In [13]:
from pydantic import BaseModel, Field
from typing import List

class MovieReview(BaseModel):
    """영화 리뷰 분석 결과"""
    title: str = Field(description='영화 제목')
    keywords: List[str] = Field(description='핵심 키워드 3개')
    summary: str = Field(description='한 문장 요약')
    sentiment: str = Field(description='감정 분석 결과 (긍정/부정/중립)')
    score: float = Field(description='평점 (별 1개~5개)')

# with_structured_output으로 Pydantic 모델 바인딩
structured_llm = gpt.with_structured_output(MovieReview)
# 출력 형식에 대한 시스템 프롬프트 삽입 + 자동 파싱
review = structured_llm.invoke(
    '매드 맥스: 분노의 도로는 액션 영화의 새로운 기준을 세운 작품입니다. 조지 밀러 감독이 '
    '70대에 만들었다는 사실이 믿기지 않을 만큼 에너지가 넘쳤어요. 황량한 사막을 가로지르는 '
    '추격전이 영화의 거의 전부를 차지하지만, CG에 의존하지 않고 실제 차량과 스턴트로 '
    '찍어낸 장면들은 압도적인 현장감을 줍니다. 대사를 최소화하고 이미지와 움직임만으로 '
    '서사를 끌고 가는 연출이 인상적이었고, 퓨리오사라는 강렬한 캐릭터를 통해 단순한 '
    '액션을 넘어선 메시지까지 담아냈습니다. 다만 쉴 틈 없이 몰아치는 전개 탓에 인물의 '
    '내면을 들여다볼 여유가 부족했던 점은 아쉬웠습니다. 그럼에도 극장에서 다시 보고 싶은, '
    '체험에 가까운 영화였습니다.'
)
print(f'타입: {type(review).__name__}')
print(f'제목: {review.title}')
print(f'키워드: {review.keywords}')
print(f'요약: {review.summary}')
print(f'감정: {review.sentiment}')
print(f'평점: {review.score}')

review

타입: MovieReview
제목: 매드 맥스: 분노의 도로
키워드: ['실제 스턴트', '추격전', '퓨리오사']
요약: 실제 차량·스턴트 기반의 압도적 추격전과 이미지 중심 연출로 액션의 새 기준을 세우면서도, 숨 가쁜 전개로 인물 내면의 여백은 다소 부족한 체험형 영화다.
감정: 긍정
평점: 4.5


MovieReview(title='매드 맥스: 분노의 도로', keywords=['실제 스턴트', '추격전', '퓨리오사'], summary='실제 차량·스턴트 기반의 압도적 추격전과 이미지 중심 연출로 액션의 새 기준을 세우면서도, 숨 가쁜 전개로 인물 내면의 여백은 다소 부족한 체험형 영화다.', sentiment='긍정', score=4.5)

### 3-2. 중첩된 Structured Output

Pydantic 클래스를 중첩하면 복잡한 구조의 데이터도 생성할 수 있습니다.

In [14]:
class DailySchedule(BaseModel):
    """하루 일정"""
    morning: str = Field(description='오전 일정')
    afternoon: str = Field(description='오후 일정')
    evening: str = Field(description='저녁 일정')

class TravelPlan(BaseModel):
    """여행 계획"""
    destination: str = Field(description='목적지')
    duration: str = Field(description='여행 기간')
    schedule: List[DailySchedule] = Field(description='일별 일정')

In [15]:
trip_planner = gpt.with_structured_output(TravelPlan)

trip_prompt = ChatPromptTemplate([
    ('system', '당신은 여행 계획 전문가입니다. 주어진 조건에 맞는 여행 계획을 작성하세요.'),
    ('human', '{destination}에서 {duration} 여행 계획을 세워주세요.')
])

# 프롬프트 + Structured Output LLM 체인
trip_chain = trip_prompt | trip_planner
plan = trip_chain.invoke({'destination': '제주도', 'duration': '2일'})

print(f'목적지: {plan.destination}')
print(f'기간: {plan.duration}')
for i, day in enumerate(plan.schedule, 1):
    print(f'\n{i}일차:')
    print(f'  오전: {day.morning}')
    print(f'  오후: {day.afternoon}')
    print(f'  저녁: {day.evening}')

목적지: 제주도
기간: 2일

1일차:
  오전: 제주공항 도착 → 용두암 산책(바다 전망) → 동문시장(간단한 아침/간식, 오메기떡·회/해산물 구경)
  오후: 애월 해안도로 드라이브(한담해변~곽지해수욕장) → 애월 카페 거리에서 휴식 → 협재해변 산책 및 사진 포인트
  저녁: 제주시 또는 애월에서 흑돼지/해산물 저녁 → 이호테우 해변(말등대 야경) 산책 → 숙소 체크인 및 휴식

2일차:
  오전: 성산일출봉(이른 산책/등반) → 섭지코지 해안 산책
  오후: 우도(선택: 왕복 배편) 자전거/스쿠터로 해안 일주, 서빈백사·우도봉 전망 → 성산으로 복귀 → 함덕해변 또는 김녕해변에서 여유
  저녁: 서귀포 이동(시간 여유 시) → 천지연폭포 또는 새연교 야경 산책 → 공항 이동 및 출발(또는 제주시 복귀 후 마무리)


---
## 4. Tool Calling

LLM Agent의 가장 주요한 기능은 도구를 호출하는 Tool Calling입니다.   

이는 명령어와 같은 형식의 구조화된 텍스트를 출력해, 해당 도구의 실행을 자동화하는 방식입니다.  

이 Tool Calling은 Structured Output의 특별한 형태에 해당합니다. 

LangChain에서는 `@tool` 데코레이터를 사용하여 파이썬 함수를 LLM이 사용할 수 있는 도구로 변환합니다.

tool의 정보는 LLM에 프롬프트를 통해 전달되어, 필요한 순간에 LLM이 도구를 사용할 수 있게 합니다.   

**아래의 정보가 LLM의 시스템 프롬프트에 포함됩니다.**

- 도구의 이름
- 입력 변수 형식
- 설명(docstring)

In [16]:
from langchain.tools import tool
from datetime import datetime

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다.

    Args:
        a: 첫 번째 정수
        b: 두 번째 정수
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다.

    Args:
        a: 첫 번째 정수
        b: 두 번째 정수
    """
    return a * b

@tool
def get_current_date() -> str:
    """현재 날짜를 YYYY-MM-DD 형식으로 반환합니다."""
    return datetime.now().strftime('%Y-%m-%d')

# 도구 직접 실행 테스트
print(f"add(3, 5) = {add.invoke({'a': 3, 'b': 5})}")
print(f"multiply(4, 7) = {multiply.invoke({'a': 4, 'b': 7})}")
print(f"get_current_date() = {get_current_date.invoke({})}")

add(3, 5) = 8
multiply(4, 7) = 28
get_current_date() = 2026-06-24


### 4-1. 도구 정보 확인

LLM은 도구의 이름, 설명, 파라미터 스키마를 통해 어떤 도구를 언제 사용할지 결정합니다.

In [17]:
print(f'도구 이름: {add.name}')
print(f'도구 설명: {add.description}')
print(f'\n파라미터 스키마 (JSON Schema):')
add.args_schema.model_json_schema()

도구 이름: add
도구 설명: 두 정수를 더합니다.

    Args:
        a: 첫 번째 정수
        b: 두 번째 정수

파라미터 스키마 (JSON Schema):


{'description': '두 정수를 더합니다.\n\nArgs:\n    a: 첫 번째 정수\n    b: 두 번째 정수',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'add',
 'type': 'object'}

---
## 5. Tool Calling 메커니즘



LLM에 도구를 연결하면, LLM은 사용자의 질문에 따라 적절한 도구를 선택하고 호출 인자를 생성합니다.

LLM의 역할은 도구를 실행하기 위한 구조화된 텍스트를 생성하는 것입니다.   
이는 `name`, `args`를 포함합니다.

### 5-1. LLM에 도구 연결 (bind_tools)

In [18]:
tools = [add, multiply, get_current_date]

# bind_tools: LLM에 도구 정보를 프롬프트에 포함
llm_with_tools = gpt.bind_tools(tools)

# with_structured_output 와 유사
print(llm_with_tools)
llm_with_tools

bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-5.2', 'release_date': '2025-12-11', 'last_updated': '2025-12-11', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002967DF81F90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002967E8F1850>, root_client=<openai.OpenAI obje

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-5.2', 'release_date': '2025-12-11', 'last_updated': '2025-12-11', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002967DF81F90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002967E8F1850>, root_client=<

### 5-2. Tool Call 메시지 분석

LLM이 도구 사용이 필요하다고 판단하면, `AIMessage`의 `tool_calls` 필드에 Tool Call 정보를 출력합니다.

In [20]:
from rich import print as rprint

messages = [HumanMessage('3과 5를 더해주세요.')]

response = llm_with_tools.invoke(messages)

rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 20,
            'prompt_tokens': 214,
            'total_tokens': 234,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.2-2025-12-11',
        'system_fingerprint': None,
        'id': 'chatcmpl-Du786FD6jzODHzDXDnWrLGJ2Ge4LC',
        'service_tier': 'default',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019ef757-b7c7-74c2-bd56-f14498611836-0',
    tool_calls=[
        {'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_nuf1NTMggBwGSbmLGSYQ9hHN', 'type': 'tool_call'}
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 214,
        'output_tokens': 20,
        'total_tokens': 234,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [ ]:
# Tool Call 상세 분석
if response.tool_calls:
    tool_call = response.tool_calls[0]
    print('--- Tool Call 구조 ---')
    print(f'  name (도구 이름): {tool_call["name"]}')
    print(f'  args (호출 인자): {tool_call["args"]}')
    print(f'  id (호출 ID):     {tool_call["id"]}')
    print(f'  type:             {tool_call["type"]}')

### 5-3. 도구 실행 및 결과 전달

Tool Call을 받으면 해당 도구를 실행하고, 결과를 `ToolMessage`로 LLM에 다시 전달합니다.

In [ ]:
# 도구 이름 -> 도구 함수 매핑
tools_map = {t.name: t for t in tools}
print(f'등록된 도구: {list(tools_map.keys())}')

# Tool Call의 결과를 ToolMessage로 변환
tool_result = tools_map[tool_call['name']].invoke(tool_call)
print(f'\nToolMessage: {tool_result}')
print(f'결과값: {tool_result.text}')
print(f'tool_call_id: {tool_result.tool_call_id}')

In [ ]:
final_result = llm_with_tools.invoke(messages + [response] + [tool_result])

final_result

### 5-4. Tool Calling 전체 흐름

Tool Calling은 다음과 같은 메시지 순서로 동작합니다:

```
HumanMessage (사용자 질문)
    → AIMessage + tool_calls (LLM이 도구 호출 결정)
        → ToolMessage (도구 실행 결과)
            → AIMessage (최종 응답)
```

LLM이 여러 도구를 동시에 호출할 수도 있습니다 (Parallel Tool Calling).

In [ ]:
from langchain.messages import HumanMessage

# Step 1: 사용자 질문
messages = [HumanMessage(content='오늘 날짜를 알려주고, 7 곱하기 8도 계산해줘.')]

# Step 2: LLM이 Tool Call 생성
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

print(f'LLM이 요청한 Tool Calls: {len(ai_msg.tool_calls)}개')
for tc in ai_msg.tool_calls:
    print(f'  -> {tc["name"]}({tc["args"]})')

In [ ]:
# Step 3: 각 Tool Call 실행 후 메시지에 추가
for tc in ai_msg.tool_calls:
    result = tools_map[tc['name']].invoke(tc)
    messages.append(result)
    print(f'{tc["name"]} 실행 결과: {result.text}')

# Step 4: 최종 응답 생성
final = llm_with_tools.invoke(messages)
print(f'\n최종 응답: {final.text}')

### 5-5. 메시지 흐름 시각화

위 과정의 전체 메시지 목록을 확인해봅시다.

In [ ]:
messages.append(final)

for i, msg in enumerate(messages):
    msg_type = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        calls = ', '.join([f"{tc['name']}({tc['args']})" for tc in msg.tool_calls])
        print(f'[{i}] {msg_type}: tool_calls=[{calls}]')
    elif hasattr(msg, 'tool_call_id'):
        print(f'[{i}] {msg_type}: ({msg.name}) -> {msg.text}')
    else:
        content_preview = msg.text[:80] + '...' if len(msg.text) > 80 else msg.text
        print(f'[{i}] {msg_type}: {content_preview}')

### 5-6. while 구문을 이용한 도구 호출 루프

실제 작업은 한 번의 도구 호출로 끝나지 않는 경우가 많습니다.    
도구 실행 결과를 확인한 뒤에 LLM이 다음 도구를 또 호출할 수도 있습니다.

다단계의 Tool Calling은 반복문을 통해 구현해 보겠습니다.

LLM의 출력인 `AIMessage`에 tool call이 있으면 도구를 실행하고 Context에 추가합니다.   
이후 다시 LLM을 호출합니다.     

In [ ]:
from langchain.messages import HumanMessage

messages = [HumanMessage('3과 5를 더하고, 거기에 10을 곱한 결과를 알려줘')]
print('HumanMessage:', messages[-1].text)

max_turns = 5  # 무한 루프 방지를 위한 상한
turn = 0

while turn < max_turns:
    turn += 1
    ai_msg = llm_with_tools.invoke(messages)
    print('---- LLM 입력 ----')
    messages.append(ai_msg)
    # AIMessage Context에 추가
    print('AIMessage:', messages[-1].text)
    
    # 호출할 도구가 없으면 루프 종료
    if not ai_msg.tool_calls:
        break

    print(f'[Tool Call] ', end='')
    for tc in ai_msg.tool_calls:
        print(f'{tc["name"]}({tc["args"]})', end=', ')
    print('\n')    
    for tc in ai_msg.tool_calls:
        result = tools_map[tc['name']].invoke(tc)
        messages.append(result)    
        print('ToolMessage:', result.text)
        # 각 도구를 실행하고 결과를 Context에 추가

---
## 6. 새로운 도구 연결하기

다양한 함수를 툴 콜링의 문법으로 연결할 수 있습니다.

In [ ]:
@tool
def sleep_recommendation(age: int) -> str:
    """나이를 입력받아 권장 수면 시간을 안내합니다.

    Args:
        age: 나이 (정수)
    """
    if age < 6:
        hours = "10-13시간"
    elif age < 13:
        hours = "9-11시간"
    elif age < 18:
        hours = "8-10시간"
    elif age < 65:
        hours = "7-9시간"
    else:
        hours = "7-8시간"

    return f"권장 수면 시간: {hours}"


# 도구 직접 실행
print(sleep_recommendation.invoke({'age': 30}))

In [ ]:
# LLM에 연결하여 Tool Calling 전체 흐름 실행
my_tools = [sleep_recommendation, add, get_current_date]
llm_with_tools = gpt.bind_tools(my_tools)

msgs = [HumanMessage('제가 31세 남성인데요, 하루에 6시간 정도 자는데 충분할까요?')]
ai_response = llm_with_tools.invoke(msgs)
msgs.append(ai_response)

# 도구 실행
my_tools_map = {t.name: t for t in my_tools}
for tc in ai_response.tool_calls:
    result = my_tools_map[tc['name']].invoke(tc)
    msgs.append(result)
    print(f'도구 실행: {tc["name"]} -> {result.text}')

# 최종 응답
final = llm_with_tools.invoke(msgs)
print(f'\n최종 응답: {final.text}')

---
## [실습] 커스텀 도구를 구성하여 Tool Calling 해보기

아래 조건에 맞는 커스텀 도구를 만들고, Tool Calling 전체 흐름을 실행해보세요.

요구사항:
1. 도구 1개 이상 새로 정의 (예: 환율 계산기, 온도 변환기, 텍스트 요약 등)
2. LLM에 `bind_tools`로 연결
3. Tool Calling 전체 흐름 실행 (HumanMessage - AIMessage - ToolMessage - 최종 응답)

도구의 docstring을 활용하여, LLM에게 해당 도구의 사용법을 전달하세요.

In [ ]:
# tool 문법으로 도구 정의
# bind_tools로 도구 연결
# invoke로 실행

---
## 정리

이번 실습에서 학습한 내용을 정리합니다.

- init_chat_model: 모델 이름만으로 다양한 LLM Provider를 통일된 인터페이스로 초기화
- Prompt Template: `ChatPromptTemplate`으로 재사용 가능한 시스템/사용자 프롬프트 구성
- 체인(Chain): `prompt | llm` 파이프 연산자로 프롬프트와 LLM을 연결하는 간결한 문법
- Structured Output: `with_structured_output()`으로 Pydantic 모델 기반 구조화된 데이터 생성
- @tool 데코레이터: 파이썬 함수를 LLM이 사용할 수 있는 도구로 변환. 이름과 docstring이 중요
- bind_tools: LLM에 도구 정보를 JSON Schema로 전달. LLM은 이를 바탕으로 도구 선택 및 인자 생성
- Tool Calling 흐름: `HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage(최종 응답)`, 구조화 출력의 특별한 사례